In [9]:
"""
Global Superstore - Data Cleaning
Input : superstore.csv (51,290 rows, 27 columns)
Output: superstore_clean.csv (ready for Tableau)
"""
 
import pandas as pd

In [10]:
# 1. Load

df = pd.read_csv("../data/superstore.csv")  # change path if needed
print("Raw shape:", df.shape)

Raw shape: (51290, 27)


In [11]:
# 2. Rename columns: the Chinese column "记录数" means "record count" (always 1)

df = df.rename(columns={"记录数": "Record.Count"})
 
# Standardise names (Customer.ID -> customer_id, etc.)

df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace('.', '_', regex=False)
)

In [12]:
# 3. Convert date columns

df["order_date"] = pd.to_datetime(df["order_date"])
df["ship_date"] = pd.to_datetime(df["ship_date"])

In [13]:
# 4. Basic quality checks

print("\nNull values:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate row_id:", df["row_id"].duplicated().sum())
print("Ship date before order date:", (df["ship_date"] < df["order_date"]).sum())


Null values:
 Series([], dtype: int64)
Duplicate rows: 0
Duplicate row_id: 0
Ship date before order date: 0


In [14]:
# 5. Drop duplicates and useless columns

df = df.drop_duplicates()
df = df.drop(columns=["record_count"])  # constant value of 1

In [15]:
# 6. Clean text columns

for col in df.columns:
    if pd.api.types.is_string_dtype(df[col]) or df[col].dtype == "object":
        df[col] = df[col].str.strip()

In [16]:
# 7. Feature engineering (new columns for analysis)

df["order_year"] = df["order_date"].dt.year
df["order_month"] = df["order_date"].dt.month
df["order_month_name"] = df["order_date"].dt.strftime("%b")
df["order_quarter"] = "Q" + df["order_date"].dt.quarter.astype(str)
df["order_year_month"] = df["order_date"].dt.to_period("M").astype(str)
df["day_of_week"] = df["order_date"].dt.day_name()
 
df["shipping_days"] = (df["ship_date"] - df["order_date"]).dt.days
df["profit_margin_pct"] = (df["profit"] / df["sales"] * 100).round(2)
df["is_loss"] = (df["profit"] < 0).map({True: "Loss", False: "Profit"})
 
# Discount buckets: useful for the discount vs profit analysis

df["discount_band"] = pd.cut(
    df["discount"],
    bins=[-0.01, 0, 0.1, 0.2, 0.3, 0.5, 1],
    labels=["No discount", "1-10%", "11-20%", "21-30%", "31-50%", "50%+"],
)

In [17]:
# 8. Drop the duplicate year column (we created order_year from the date)

df = df.drop(columns=["year"])

In [18]:
# 9. Save

df.to_csv("superstore_clean.csv", index=False)
print("\nClean shape:", df.shape)
print(df.head())


Clean shape: (51290, 35)
          category         city        country customer_id     customer_name  \
0  Office Supplies  Los Angeles  United States   LS-172304  Lycoris Saunders   
1  Office Supplies  Los Angeles  United States   MV-174854     Mark Van Huff   
2  Office Supplies  Los Angeles  United States   CS-121304      Chad Sievert   
3  Office Supplies  Los Angeles  United States   CS-121304      Chad Sievert   
4  Office Supplies  Los Angeles  United States   AP-109154    Arthur Prichep   

   discount market order_date        order_id order_priority  ... order_year  \
0       0.0     US 2011-01-07  CA-2011-130813           High  ...       2011   
1       0.0     US 2011-01-21  CA-2011-148614         Medium  ...       2011   
2       0.0     US 2011-08-05  CA-2011-118962         Medium  ...       2011   
3       0.0     US 2011-08-05  CA-2011-118962         Medium  ...       2011   
4       0.0     US 2011-09-29  CA-2011-146969           High  ...       2011   

  order_mont